## 1. Importing libraries

In [6]:
import duckdb
import numpy as np

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from detector.eval.metrics import summarise, threshold_at_fpr
from detector.features.text_clean import strip_artifacts

In [7]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = duckdb.connect().sql(f"select * from '{ROOT / 'data/processed/essays.parquet'}'").df()

#the test split stays sealed until I compare final models
train, val = df[df.split == "train"], df[df.split == "val"]
len(train), len(val)

(14171, 3036)

In [9]:
def fit(texts, y):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
    X = vec.fit_transform(texts)
    clf = LogisticRegression(max_iter=1_000, C=10).fit(X, y)
    return vec, clf

def score(vec, clf, texts):
    return clf.predict_proba(vec.transform(texts))[:, 1]

vec, clf = fit(train.text, train.label)
val_scores = score(vec, clf, val.text)

In [11]:
for fpr in (.01, .001):
    thr = threshold_at_fpr(val.label, val_scores, fpr)
    print(f"budget {fpr}:", summarise(val.label, val_scores, thr))

budget 0.01: {'roc_auc': 0.9999992010533313, 'pr_auc': 0.9999958772407197, 'tpr': 1.0, 'fpr': 0.009827044025157232}
budget 0.001: {'roc_auc': 0.9999992010533313, 'pr_auc': 0.9999958772407197, 'tpr': 1.0, 'fpr': 0.0007861635220125787}


In [12]:
#lets strip off the formatting and placeholds to see if the score depends on them
val_clean = val.text.map(strip_artifacts)
thr = threshold_at_fpr(val.label, val_scores, 0.01)

scores_raw_clean = score(vec, clf, val_clean)
print("trained raw, tested clean:", summarise(val.label, scores_raw_clean, thr))

vec_c, clf_c = fit(train.text.map(strip_artifacts), train.label)
scores_clean = score(vec_c, clf_c, val_clean)
thr_c = threshold_at_fpr(val.label, scores_clean, 0.01)
print("trained clean, tested clean:", summarise(val.label, scores_clean, thr_c))

trained raw, tested clean: {'roc_auc': 0.9999976031599939, 'pr_auc': 0.9999876816949987, 'tpr': 1.0, 'fpr': 0.009827044025157232}
trained clean, tested clean: {'roc_auc': 0.9999976031599939, 'pr_auc': 0.9999876816949987, 'tpr': 1.0, 'fpr': 0.009827044025157232}


In [13]:
names = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
print("most AI-like:    ", list(names[order[-15:]][::-1]))
print("most human-like: ", list(names[order[:15]]))

most AI-like:     ['conclusion', 'potential', 'in conclusion', 'benefits', 'it not', 'support', 'it like', 'climate', 'perspectives', 'privacy', 'significant', 'health', 'name', 'the potential', 're']
most human-like:  ['because', 'would', 'then', 'very', 'people', 'going', 'many', 'do', 'because of', 'go', 'paragraph', 'will', 'percent', 'was', 'he']
